In [0]:
# Create silver schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS 02_silver_catalog.silver_schema")
print("✓ Silver schema created/verified\n")

# Define the cleaned bronze tables to copy to silver
cleaned_tables = [
    "crm_cust_base_clean",
    "inv_levels_clean",
    "mst_product_master_clean",
    "raw_sales_dtl_clean",
    "rtn_trans_logs_clean",
    "stores_geo_list_final_clean"
]

print("Starting silver layer insertion...\n")
print("=" * 60)

# Copy each cleaned table from bronze to silver
for table_name in cleaned_tables:
    source_table = f"01_bronze_catalog.bronze_stage_schema.{table_name}"
    target_table = f"02_silver_catalog.silver_schema.{table_name}"
    
    # Read from bronze cleaned table
    df = spark.read.table(source_table)
    row_count = df.count()
    
    # Write to silver catalog
    df.write.mode("overwrite").saveAsTable(target_table)
    
    print(f"✓ {table_name}")
    print(f"  Rows: {row_count:,} | Columns: {len(df.columns)}")
    print()

print("=" * 60)
print("\n✅ All cleaned tables successfully inserted into silver catalog!")
print(f"\nLocation: 02_silver_catalog.silver_schema")
print(f"Total tables: {len(cleaned_tables)}")

In [0]:
# Verify tables in silver catalog
print("\n" + "=" * 60)
print("SILVER CATALOG - ALL TABLES")
print("=" * 60)

silver_tables = spark.sql("SHOW TABLES IN 02_silver_catalog.silver_schema").collect()

print(f"\nTotal tables in silver schema: {len(silver_tables)}\n")

for table in silver_tables:
    table_name = table.tableName
    df = spark.read.table(f"02_silver_catalog.silver_schema.{table_name}")
    row_count = df.count()
    col_count = len(df.columns)
    print(f"  ✓ {table_name:<35} | Rows: {row_count:>4} | Cols: {col_count}")

print("\n" + "=" * 60)
print("Sample from silver - crm_cust_base_clean")
print("=" * 60)

# Display sample data from one silver table
# df_sample = spark.read.table("02_silver_catalog.silver_schema.crm_cust_base_clean")
# display(df_sample)



In [0]:
df_sample = spark.read.table("02_silver_catalog.silver_schema.crm_cust_base_clean")
display(df_sample)